In [3]:
import pandas as pd

file_path = "../Differential_privacy/PrivFill/data/bbc.csv"
df = pd.read_csv(file_path)
print(df.head(10))

                                                text          label
0  Ray DVD beats box office takings Oscar-nominat...  entertainment
1  Brazil plays down Varig rescue The Brazilian g...       business
2  Campbell rescues Arsenal Sol Campbell proved t...          sport
3  'Blog' picked as word of the year The term "bl...           tech
4  Virgin Blue shares plummet 20% Shares in Austr...       business
5  UKIP MEP attacked German 'empire' A UK Indepen...       politics
6  Criminal probe on Citigroup deals Traders at U...       business
7  O'Driscoll saves Irish blushes Two moments of ...          sport
8  EA to take on film and TV giants Video game gi...           tech
9  Early Elvis recordings go on sale Some of Elvi...  entertainment


In [4]:
df = df.drop_duplicates(subset='text')
df['message_length'] = df['text'].apply(len)
df = df.sort_values(by='message_length', ascending=False)
df_2 = df[df['message_length'] < 5000]
print(df_2.head(10))

                                                   text     label  \
1822  Gardener battles to narrow win Jason Gardener ...     sport   
2024  Peer-to-peer nets 'here to stay' Peer-to-peer ...      tech   
51    Peer-to-peer nets 'here to stay' Peer-to-peer ...      tech   
310   The 'ticking budget' facing the US The budget ...  business   
112   Cebit fever takes over Hanover Thousands of pr...      tech   
335   Technology gets the creative bug The hi-tech a...      tech   
69    Technology gets the creative bug The hi-tech a...      tech   
1084  Howard's unfinished business "He's not finishe...  politics   
990   TV's future down the phone line Internet TV ha...      tech   
166   UKIP's secret weapon? By any measure, New York...  politics   

      message_length  
1822            4963  
2024            4946  
51              4945  
310             4936  
112             4904  
335             4894  
69              4893  
1084            4891  
990             4887  
166       

In [5]:
import json
from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()
def count_entities(text):
    st_analyze_results = analyzer.analyze(
        text=text,
        language="en",
        score_threshold=0.5,
        allow_list=[],
    )
    # results_as_dicts = [result.to_dict() for result in st_analyze_results]
    # results_json = json.dumps(results_as_dicts, indent=2)
    # print(results_json)
    return len(st_analyze_results)

df_2['pii_nr'] = df_2['text'].apply(count_entities)
df_2 = df_2.sort_values(by='pii_nr', ascending=False)
print(df_2.head(20))

                                                   text          label  \
1815  Dawson wins England squad recall Wasps scrum-h...          sport   
1453  O'Driscoll/Gregan lead Aid stars Ireland's Bri...          sport   
3114  Italy 8-38 Wales Wales secured their first awa...          sport   
1679  Wales win in Rome Wales secured their first aw...          sport   
1100  Ireland call up uncapped Campbell Ulster scrum...          sport   
537   Ireland 19-13 England Ireland consigned Englan...          sport   
1725  Ireland surge past Scots Ireland maintained th...          sport   
94    Chelsea clinch cup in extra-time (after extra-...          sport   
2951  Britain boosted by Holmes double Athletics fan...          sport   
416   England 17-18 France England suffered an eight...          sport   
152   Ireland 17-12 South Africa Ronan O'Gara scored...          sport   
2     Campbell rescues Arsenal Sol Campbell proved t...          sport   
2457  Taylor poised for Scotland retur

/var/folders/mv/q7zzt95s3bjbxyj60fgc67lh0000gn/T/ipykernel_14094/3313464642.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['pii_nr'] = df_2['text'].apply(count_entities)


In [6]:
df_3 = df_2[df_2['pii_nr'] < 100]
df_3 = df_3[df_3['pii_nr'] > 25]
df_3 = df_3[df_3['message_length'] > 2000]
print(len(df_3))

574


In [7]:
print(df_3.head(50))

                                                   text          label  \
1100  Ireland call up uncapped Campbell Ulster scrum...          sport   
537   Ireland 19-13 England Ireland consigned Englan...          sport   
1725  Ireland surge past Scots Ireland maintained th...          sport   
94    Chelsea clinch cup in extra-time (after extra-...          sport   
2951  Britain boosted by Holmes double Athletics fan...          sport   
416   England 17-18 France England suffered an eight...          sport   
152   Ireland 17-12 South Africa Ronan O'Gara scored...          sport   
2     Campbell rescues Arsenal Sol Campbell proved t...          sport   
1848  Reds sink 10-man Magpies Titus Bramble's own g...          sport   
721   Holmes back on form in Birmingham Double Olymp...          sport   
1934  The Producers scoops stage awards The Producer...  entertainment   
7     O'Driscoll saves Irish blushes Two moments of ...          sport   
1661  Federer joins all-time greats Th

In [8]:
def first_four_words(text):
    return ' '.join(text.split()[:4])
df_4 = df_3
df_4['first_four_words'] = df_4['text'].apply(first_four_words)
df_4 = df_4.drop_duplicates(subset='first_four_words')
len(df_4)

513

In [ ]:
df_4['text'].to_csv("./BBC_preprocessed.csv", index=False)